<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Processing Fundamental — Implementation</b></h1>
</div>

This notebook executes the core numerical image-processing experiments covering image representation, pixel and ROI operations, channel conventions, dynamic range, noise models, image-comparison metrics, encoding effects, and validation.


## Setup — Environment and Configuration

Import the required libraries and seed the random generator for reproducible experiments.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Fix the RNG so every noise experiment is reproducible.
RNG = np.random.default_rng(42)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths

Resolve repository-relative inputs and prepare the figure output directory.


In [ ]:
# Resolve the lab root robustly whether execution starts from root or notebooks/.
def find_lab_root(start: Path) -> Path:
    """Locate the nearest valid lab root from the current execution directory.

    Searching upward makes the notebook portable when launched either from the
    lab directory or from notebooks/, while requiring both data/ and notebooks/
    prevents accidentally selecting an unrelated parent directory.
    """
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the lab root. "
        "Expected a directory containing both 'data/' and 'notebooks/'."
    )


LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "einstein": DATA_DIR / "einstein.png",
    "peppers": DATA_DIR / "peppers.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "grass": DATA_DIR / "grass.jpg",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
}

missing_files = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing_files, f"Missing input files: {missing_files}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

## 2. Sampling and Quantization Experiments

Run controlled sampling and quantization experiments and save the required diagnostics.


### Sampling Experiment


In [ ]:
# Use a smooth synthetic field to isolate spatial-sampling effects.
x = np.linspace(0, 2 * np.pi, 256)
y = np.linspace(0, 2 * np.pi, 256)
xx, yy = np.meshgrid(x, y)

continuous_like = (
    0.55
    + 0.25 * np.sin(2.0 * xx)
    + 0.20 * np.cos(3.0 * yy)
)
continuous_like = np.clip(continuous_like, 0.0, 1.0)

# Use progressively coarser powers-of-four sampling to make spatial information loss obvious.
sampling_steps = [1, 4, 8, 16]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))

for ax, step in zip(axes, sampling_steps):
    sampled = continuous_like[::step, ::step]
    ax.imshow(sampled, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Sampling step = {step}\nshape = {sampled.shape}")
    ax.axis("off")

fig.suptitle("Spatial Sampling")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_sampling.png", dpi=300, bbox_inches="tight")
plt.show()

### Quantization Experiment


In [ ]:
gradient = np.tile(np.linspace(0, 1, 512), (90, 1))

# Span binary to full 8-bit quantization so banding disappears progressively.
bit_depths = [1, 2, 4, 8]

fig, axes = plt.subplots(4, 1, figsize=(11, 6))

for ax, bits in zip(axes, bit_depths):
    levels = 2 ** bits

    # Map normalized intensities to a controlled number of quantization levels.
    quantized = np.round(gradient * (levels - 1)) / (levels - 1)

    ax.imshow(quantized, cmap="gray", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"{bits}-bit quantization → {levels} intensity levels")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_quantization.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

The sampling experiment separates **spatial resolution** from **intensity resolution**. Reducing the number of spatial samples removes fine geometric detail and can introduce aliasing, whereas reducing the number of quantization levels preserves geometry but produces visible intensity banding. The two degradations therefore arise from different stages of digital image formation and should not be interpreted as the same type of information loss.


## 3. Pixel Coordinates and Array Representation

Verify the row/column coordinate convention on a controlled grayscale matrix.


In [ ]:
# Use a small controlled image so pixel values and neighborhoods remain inspectable.
toy_gray = np.array(
    [
        [0, 32, 64, 96, 128],
        [24, 56, 88, 120, 152],
        [48, 80, 112, 144, 176],
        [72, 104, 136, 168, 208],
        [96, 128, 160, 208, 255],
    ],
    dtype=np.uint8,
)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(toy_gray, cmap="gray", vmin=0, vmax=255)

for row in range(toy_gray.shape[0]):
    for col in range(toy_gray.shape[1]):
        ax.text(
            col,
            row,
            str(toy_gray[row, col]),
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("A grayscale image is a matrix of intensities")
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_grayscale_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("Shape:", toy_gray.shape)
print("dtype:", toy_gray.dtype)
print("Pixel at row=2, column=3:", toy_gray[2, 3])

## 4. Image Representation Modes

Construct representative image types and compare their numerical representations.


In [ ]:
# Contrast binary, grayscale and RGB storage with simple controlled examples.
binary = np.zeros((120, 160), dtype=np.uint8)
binary[30:90, 45:120] = 255

grayscale = np.tile(
    np.linspace(0, 255, 160, dtype=np.uint8),
    (120, 1),
)

rgb = np.zeros((120, 160, 3), dtype=np.uint8)
rgb[:, :53] = [255, 0, 0]
rgb[:, 53:106] = [0, 255, 0]
rgb[:, 106:] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))

axes[0].imshow(binary, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Binary\nshape={binary.shape}")

axes[1].imshow(grayscale, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale\nshape={grayscale.shape}")

axes[2].imshow(rgb)
axes[2].set_title(f"RGB\nshape={rgb.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_image_types.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Load and Inspect Real Images

Load the supplied images and report their basic numerical properties.


In [ ]:
# Load every reference image through one consistent RGB conversion path.
images = {}

for name, path in IMAGE_FILES.items():
    images[name] = np.asarray(Image.open(path).convert("RGB"))

for name, image in images.items():
    print(
        f"{name:9s} | "
        f"shape={str(image.shape):16s} "
        f"dtype={image.dtype} "
        f"range=[{image.min()}, {image.max()}]"
    )

In [ ]:
# Inspect all reference images together before applying any processing.
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Dimensions, Resolution, Aspect Ratio, and Channels

Measure spatial dimensions, pixel counts, channels, and aspect ratios for the reference images.


In [ ]:
# Derive dimensions, channel count and aspect ratio from one representative RGB image.
peppers = images["peppers"]

height, width, channels = peppers.shape
pixel_count = height * width
aspect_ratio = width / height

print(f"Height       : {height} pixels")
print(f"Width        : {width} pixels")
print(f"Channels     : {channels}")
print(f"Pixel count  : {pixel_count:,}")
print(f"Aspect ratio : {aspect_ratio:.3f}")

### Interpretation

Image shape and channel count determine how later operations must interpret the array. A grayscale image is a 2-D intensity field, while an RGB image contains three aligned channels. Preserving the aspect ratio avoids geometric distortion, and explicitly checking dimensions prevents silent errors when comparing, indexing, or combining images.


## 7. Data Types, Bit Depth, Dynamic Range, and Memory

Compare datatype ranges, memory usage, and observed image dynamic ranges.


In [ ]:
# Connect dtype and item size to legal range and in-memory storage cost.
print("dtype:", peppers.dtype)
print("bytes per value:", peppers.dtype.itemsize)
print("array memory:", f"{peppers.nbytes:,} bytes")
print("array memory:", f"{peppers.nbytes / 1024**2:.3f} MiB")

uint8_info = np.iinfo(np.uint8)
print("uint8 range:", uint8_info.min, "to", uint8_info.max)

## 8. Display Scaling and Visualization Control

Compare automatic and fixed display scaling without modifying the underlying image data.


In [ ]:
# Demonstrate why explicit display limits matter for low-contrast images.
# Restrict values to a narrow mid-gray interval to isolate display-scaling effects.
low_contrast = np.linspace(90, 165, 256, dtype=np.uint8)
low_contrast = np.tile(low_contrast, (120, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].imshow(low_contrast, cmap="gray")
axes[0].set_title("Automatic display scaling")

axes[1].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Fixed display range: 0–255")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_display_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Pixel Access and Safe Modification

Read and modify selected pixels on a copy while preserving the original array.


In [ ]:
# Edit one known pixel neighborhood without mutating the original image.
ballons = images["ballons"]

y, x = 140, 220
original_pixel = ballons[y, x].copy()

edited_ballons = ballons.copy()

radius = 6
edited_ballons[
    y - radius : y + radius + 1,
    x - radius : x + radius + 1,
] = [255, 0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(ballons)
axes[0].scatter([x], [y], s=70, facecolors="none", edgecolors="yellow")
axes[0].set_title("Original + selected pixel")

axes[1].imshow(edited_ballons)
axes[1].set_title("Edited copy")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_pixel_edit.png", dpi=300, bbox_inches="tight")
plt.show()

print("Selected coordinate (x, y):", (x, y))
print("Stored RGB value:", original_pixel)

## 10. Regions of Interest (ROI)

Extract and visualize a rectangular ROI using explicit coordinate bounds.


In [ ]:
# Extract an ROI with explicit coordinates to make spatial indexing observable.
tower = images["tower"]

y0, y1 = 120, 360
x0, x1 = 170, 390

roi = tower[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(tower)
axes[0].add_patch(
    plt.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
)
axes[0].set_title("Full image and ROI")

axes[1].imshow(roi)
axes[1].set_title(f"ROI: {roi.shape[1]}×{roi.shape[0]}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_region_of_interest.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Pixel Neighborhoods

Extract local neighborhoods and report their numerical statistics.


In [ ]:
# Apply the standard luminance weighting instead of averaging RGB channels.
einstein_rgb = images["einstein"]

einstein_gray_float = (
    0.299 * einstein_rgb[..., 0].astype(np.float32)
    + 0.587 * einstein_rgb[..., 1].astype(np.float32)
    + 0.114 * einstein_rgb[..., 2].astype(np.float32)
)
einstein_gray = np.clip(einstein_gray_float, 0, 255).astype(np.uint8)

y, x = 120, 120
patch_3x3 = einstein_gray[y - 1 : y + 2, x - 1 : x + 2]

print("Center pixel:", einstein_gray[y, x])
print("3×3 neighborhood:")
print(patch_3x3)

## 12. RGB Channel Decomposition

Separate the RGB channels and measure their individual statistics.


In [ ]:
# Decompose RGB explicitly to expose channel-specific structure.
red = peppers[..., 0]
green = peppers[..., 1]
blue = peppers[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(peppers)
axes[0].set_title("RGB")

axes[1].imshow(red, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Red channel")

axes[2].imshow(green, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Green channel")

axes[3].imshow(blue, cmap="gray", vmin=0, vmax=255)
axes[3].set_title("Blue channel")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_rgb_channels.png", dpi=300, bbox_inches="tight")
plt.show()

print(
    "Channel means:",
    {
        "R": round(float(red.mean()), 2),
        "G": round(float(green.mean()), 2),
        "B": round(float(blue.mean()), 2),
    },
)

### Interpretation

The separated RGB channels reveal that color information is distributed unevenly across the three components. Structures that are strong in one channel may be weak in another, which explains why color-aware processing can succeed where a single grayscale representation loses discriminative information.


## 13. RGB and BGR Conventions

Demonstrate the effect of channel-order mismatch and recover the correct representation.


In [ ]:
# Reverse channel order to illustrate RGB/BGR convention errors.
rgb_example = peppers
bgr_like = rgb_example[..., ::-1]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(rgb_example)
axes[0].set_title("Correct RGB")

axes[1].imshow(bgr_like)
axes[1].set_title("Channels reversed")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_rgb_bgr.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. RGB-to-Grayscale Conversion

Compare arithmetic-mean and luminance-weighted grayscale conversions.


In [ ]:
def rgb_to_grayscale(rgb_image: np.ndarray) -> np.ndarray:
    """Convert RGB uint8 data to perceptual grayscale.

    The 0.299/0.587/0.114 weights approximate luminance sensitivity, so green
    contributes more than blue. Arithmetic is promoted to float before the
    weighted sum to avoid uint8 overflow or truncation.
    """

    # Promote before arithmetic to avoid unsigned overflow.
    rgb_float = rgb_image.astype(np.float32)

    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )

    return np.clip(gray, 0, 255).astype(np.uint8)


einstein_rgb = images["einstein"]
einstein_gray = rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(einstein_rgb)
axes[0].set_title(f"RGB shape: {einstein_rgb.shape}")

axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale shape: {einstein_gray.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_rgb_to_grayscale.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

Grayscale conversion is a weighted projection of color information, not a simple removal of two channels. The resulting intensity image preserves much of the perceived luminance structure but discards chromatic differences, so algorithms operating on grayscale should only be used when color is not essential to the task.


## 15. Image Statistics

Compute global and channel-wise descriptive statistics.


In [ ]:
# Summarize intensity distribution with robust descriptive statistics.
grass_gray = rgb_to_grayscale(images["grass"])

statistics = {
    "min": int(grass_gray.min()),
    "max": int(grass_gray.max()),
    "mean": float(grass_gray.mean()),
    "median": float(np.median(grass_gray)),
    "std": float(grass_gray.std()),
    "p05": float(np.percentile(grass_gray, 5)),
    "p95": float(np.percentile(grass_gray, 95)),
}

for name, value in statistics.items():
    print(f"{name:>6s}: {value:.3f}" if isinstance(value, float) else f"{name:>6s}: {value}")

## 16. Intensity Histograms

Compute and validate 8-bit intensity histograms.


In [ ]:
# Pair the image with its histogram to connect appearance and intensity counts.
ballons_gray = rgb_to_grayscale(images["ballons"])

counts, bin_edges = np.histogram(
    ballons_gray.ravel(),
    bins=256,
    range=(0, 256),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")

axes[1].plot(np.arange(256), counts)
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_intensity_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Histogram count:", counts.sum())
print("Number of image pixels:", ballons_gray.size)

In [ ]:
# Preserve the histogram while destroying spatial arrangement to show its limitation.
shuffled = ballons_gray.ravel().copy()
RNG.shuffle(shuffled)
shuffled = shuffled.reshape(ballons_gray.shape)

hist_original, _ = np.histogram(ballons_gray.ravel(), bins=256, range=(0, 256))
hist_shuffled, _ = np.histogram(shuffled.ravel(), bins=256, range=(0, 256))

print("Histograms identical:", np.array_equal(hist_original, hist_shuffled))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(shuffled, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Pixels shuffled")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Interpretation

The histogram summarizes how pixel intensities are distributed but does not encode their spatial arrangement. Peaks indicate frequently occurring intensity ranges, while a narrow distribution often signals limited contrast. Histogram interpretation is therefore useful for exposure and contrast analysis, but it cannot by itself describe image structure.


## 17. Dynamic Range and Min-Max Normalization

Normalize the observed intensity range and compare before/after distributions.


In [ ]:
def minmax_normalize(gray_image: np.ndarray) -> np.ndarray:
    """Linearly expand the occupied gray range to [0, 255].

    This improves contrast without changing intensity ordering. A constant image
    is handled separately because its zero dynamic range makes the normalization
    denominator undefined.
    """

    image_float = gray_image.astype(np.float32)

    minimum = image_float.min()
    maximum = image_float.max()

    # Constant images have zero dynamic range; return a stable result.
    # Zero dynamic range cannot be stretched; return a stable all-zero image.
    if maximum == minimum:
        return np.zeros_like(gray_image)

    normalized = (image_float - minimum) / (maximum - minimum)
    normalized *= 255.0

    return np.clip(normalized, 0, 255).astype(np.uint8)


source_gray = rgb_to_grayscale(images["ballons"])

# Compress the source into ~30% of the 8-bit range before testing normalization.
low_contrast = 90 + (source_gray.astype(np.float32) / 255.0) * 76
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

normalized = minmax_normalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before normalization")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After min-max normalization")
axes[1, 0].axis("off")

axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After normalization")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "13_dynamic_range_normalization.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range :", int(normalized.min()), "to", int(normalized.max()))

### Interpretation

Min-max normalization expands the occupied intensity interval to the available display range. This improves numerical or visual contrast when the original image uses only a narrow portion of the range, but it does not create new information; it only remaps existing values.


## 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point

Demonstrate unsafe integer arithmetic and verify the corrected floating-point workflow.


In [ ]:
value = np.array([250], dtype=np.uint8)

# Demonstrate why unsigned arithmetic must not be used for intermediate sums.
unsafe = value + np.array([20], dtype=np.uint8)

# Perform arithmetic in a wider dtype, then clip before casting back.
safe_float = value.astype(np.float32) + 20.0
safe_uint8 = np.clip(safe_float, 0, 255).astype(np.uint8)

print("Original value :", value[0])
print("Unsafe result  :", unsafe[0])
print("Safe result    :", safe_uint8[0])

## 19. Noise Model Simulation

Generate controlled noisy observations using the specified noise models.


In [ ]:
base_gray = einstein_gray

# Use distinct noise models to expose their different statistics.
# Use zero-mean Gaussian noise so sigma alone controls additive noise strength.
# sigma=20 creates visible additive noise without completely obscuring structure.
gaussian_noise = RNG.normal(0.0, 20.0, size=base_gray.shape)
gaussian = np.clip(
    base_gray.astype(np.float32) + gaussian_noise,
    0,
    255,
).astype(np.uint8)

# Corrupt only a sparse pixel fraction to model impulse noise rather than blur.
salt_pepper = base_gray.copy()
# Corrupt 3% of pixels so impulse noise is sparse but clearly measurable.
probability = 0.03
random_map = RNG.random(base_gray.shape)
salt_pepper[random_map < probability / 2] = 0
salt_pepper[random_map > 1 - probability / 2] = 255

scaled = base_gray.astype(np.float32) / 255.0
# Scale intensities before Poisson sampling so variance follows signal level.
# Scale=30 gives visible signal-dependent variance while preserving recognizability.
poisson = RNG.poisson(scaled * 30.0) / 30.0
poisson = np.clip(poisson * 255.0, 0, 255).astype(np.uint8)

# Model speckle multiplicatively so stronger signal receives stronger perturbation.
# 0.18 multiplicative spread creates clear speckle without saturating most pixels.
speckle_noise = RNG.normal(0.0, 0.18, size=base_gray.shape)
speckle = base_gray.astype(np.float32) * (1.0 + speckle_noise)
speckle = np.clip(speckle, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))

examples = [
    ("Original", base_gray),
    ("Gaussian", gaussian),
    ("Salt & pepper", salt_pepper),
    ("Poisson", poisson),
    ("Speckle", speckle),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_noise_models.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

The simulated degradations have visibly different statistics: Gaussian noise perturbs most pixels continuously, salt-and-pepper noise produces sparse extreme values, Poisson noise depends on signal intensity, and speckle behaves multiplicatively. Because their mechanisms differ, the same restoration filter should not be expected to perform equally well on all four cases.


## 20. Image Comparison Metrics

Compute MAE, MSE, RMSE, and PSNR for the degraded observations.


In [ ]:
# Compute complementary fidelity metrics under one consistent shape contract.
def image_metrics(reference: np.ndarray, test: np.ndarray) -> dict:
    """Compute complementary full-reference distortion metrics.

    MAE measures average absolute error, MSE/RMSE emphasize larger deviations,
    and PSNR expresses MSE on a logarithmic scale. Inputs must have identical
    shape so each pixel is compared with its true counterpart.
    """

    # Reject misregistered arrays because pixel-wise metrics would be meaningless.
    if reference.shape != test.shape:
        raise ValueError("Images must have identical shapes.")

    reference_f = reference.astype(np.float64)
    test_f = test.astype(np.float64)

    difference = reference_f - test_f

    mae = np.mean(np.abs(difference))
    mse = np.mean(difference ** 2)
    rmse = np.sqrt(mse)

    # Identical images have zero MSE and therefore infinite theoretical PSNR.
    if mse == 0:
        psnr = np.inf
    else:
        psnr = 10.0 * np.log10((255.0 ** 2) / mse)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "PSNR": psnr,
    }


noise_results = {
    "Gaussian": image_metrics(base_gray, gaussian),
    "Salt & pepper": image_metrics(base_gray, salt_pepper),
    "Poisson": image_metrics(base_gray, poisson),
    "Speckle": image_metrics(base_gray, speckle),
}

for noise_name, metrics in noise_results.items():
    print(noise_name)
    for metric_name, value in metrics.items():
        print(f"  {metric_name:>4s}: {value:.4f}")

### Interpretation

MAE, MSE, RMSE, and PSNR quantify different views of numerical distortion. MSE/RMSE penalize large errors more strongly than MAE, while PSNR expresses fidelity on a logarithmic scale. These metrics are useful for controlled comparisons against a reference image, but they should be interpreted together with visual evidence because equal numerical error can correspond to different perceptual artifacts.


## 21. Lossless vs Lossy Image Encoding

Save, reload, and quantitatively compare lossless and lossy encodings.


In [ ]:
example = images["peppers"]

# Save the same image with lossless and lossy codecs for direct comparison.
png_path = OUTPUT_DIR / "15_saved_example.png"
jpg_path = OUTPUT_DIR / "15_saved_example.jpg"

Image.fromarray(example).save(png_path)
# JPEG quality 75 is a moderate-loss setting: artifacts are measurable but not extreme.
Image.fromarray(example).save(jpg_path, quality=75)

png_reload = np.asarray(Image.open(png_path).convert("RGB"))
jpg_reload = np.asarray(Image.open(jpg_path).convert("RGB"))

png_metrics = image_metrics(example, png_reload)
jpg_metrics = image_metrics(example, jpg_reload)

print("PNG reload MSE :", png_metrics["MSE"])
print("JPEG reload MSE:", jpg_metrics["MSE"])
print("PNG path :", png_path)
print("JPEG path:", jpg_path)

### Interpretation

PNG preserves the reconstructed pixel values for this workflow, whereas JPEG trades exact fidelity for compression and may introduce block/quantization artifacts. The comparison demonstrates that file format is part of the processing pipeline: re-saving an image can alter the numerical data even when no explicit image-processing operation is applied.


## 22. Standard Image Inspection Workflow

Apply the consolidated inspection procedure to every supplied reference image.


In [ ]:
# Provide one reusable inspection routine for unfamiliar image arrays.
def inspect_image(name: str, image: np.ndarray) -> None:
    """Report the minimum metadata needed to validate an unfamiliar image array.

    Shape, dimensionality, dtype, range, mean, and memory footprint are checked
    because downstream image operations depend directly on these properties.
    """

    print(f"Name       : {name}")
    print(f"Shape      : {image.shape}")
    print(f"Dimensions : {image.ndim}")
    print(f"dtype      : {image.dtype}")
    print(f"Min / max  : {image.min()} / {image.max()}")
    print(f"Mean       : {image.mean():.3f}")
    print(f"Memory     : {image.nbytes:,} bytes")


inspect_image("peppers", peppers)

## 23. Validation Checks

Run the final numerical, shape, range, metric, and output-file checks.


In [ ]:
# Validate representation, histogram, normalization, metrics and ROI invariants.
assert peppers.ndim == 3
assert peppers.shape[2] == 3
assert einstein_gray.ndim == 2

assert peppers.dtype == np.uint8
assert einstein_gray.dtype == np.uint8

assert counts.sum() == ballons_gray.size

assert normalized.min() == 0
assert normalized.max() == 255

self_metrics = image_metrics(einstein_gray, einstein_gray)
assert self_metrics["MAE"] == 0
assert self_metrics["MSE"] == 0
assert self_metrics["RMSE"] == 0
assert np.isinf(self_metrics["PSNR"])

assert roi.shape[0] == (y1 - y0)
assert roi.shape[1] == (x1 - x0)

print("All fundamental validation checks passed.")

### Interpretation

The validation stage confirms that the generated arrays have valid shapes, ranges, finite values, and expected output files. Passing these checks establishes implementation consistency; it does not replace interpretation of the images, histograms, and numerical comparisons above.


## Final Result Summary

All required experiments, figures, and validation checks are complete.